# Rush E

The "Rush E" score played as a **physics** demo with `audio` for the music, `scene3d` (with physics switched on) for the visuals.

A translucent pad floats in the middle of the scene. As the piece plays, one colored sphere is dropped for every note, timed so it falls *through* the pad at the exact moment you hear that note. Low notes drop on the left, high notes on the right, so the melody draws itself across the pad like a piano roll.

The pad sits inside a `Group`, which is how it stays put. Physics gives every loose mesh a rigid body and lets it fall, but skips grouped meshes, so the spheres pass straight through instead of bouncing off.

Press **Run All Cells** at the top of the notebook, then scroll down to the 3D scene.

In [ ]:
from codetto import scene3d, audio
import colorsys, math

scene = scene3d.Scene()
scene.set_sky(scene3d.Sky.PURE_SKY)
scene.ambient.set_brightness(60)
scene.camera.set_position(0, 4, -22).look_at(0, 1, 0)

# The translucent pad the notes fall through. It lives inside a Group so physics
# leaves it alone (grouped meshes get no rigid body), which lets the falling
# spheres pass straight through it instead of colliding with it.
PAD_Y = 0.0
pad_holder = scene3d.Group()
pad = scene3d.Shapes.Plane(width=22, height=5)
pad.set_position(0, PAD_Y, 0)
pad.set_rotation(x=90)          # lay it flat
pad.set_color("#7fdfff")
pad.set_alpha(0.18)
pad_holder.add(pad)
scene.add(pad_holder)

# Physics on: every sphere added from here on falls under this gravity.
GRAVITY = 15.0
scene.set_physics(True, gravity=(0, -GRAVITY, 0))

In [ ]:
# One column per pitch used in the piece, low note on the left, high note on
# the right — like a piano roll. Each dropped sphere is tinted for its pitch.
pitches = ['D#4', 'E4', 'F4', 'F#4', 'G#4', 'A4', 'B4', 'C5',
           'D5', 'E5', 'F5', 'G#5', 'A5', 'B5', 'C6', 'D6']

SPACING = 1.3

def pitch_x(pitch):
    i = pitches.index(pitch)
    return i * SPACING - (len(pitches) - 1) * SPACING / 2

def pitch_color(pitch):
    hue = pitches.index(pitch) / len(pitches)
    r, g, b = colorsys.hsv_to_rgb(hue, 0.85, 1.0)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

In [ ]:
#@title Note and Timing Sequence
sequence = [
  ("E4", 0.326), ("E4", 0.32), ("E4", 0.315), ("E4", 0.31), ("E4", 0.305), ("E4", 0.3),
  ("E4", 0.296), ("E4", 0.291), ("E4", 0.287), ("E4", 0.283), ("E4", 0.279), ("E4", 0.275),
  ("E4", 0.271), ("E4", 0.267), ("E4", 0.263), ("E4", 0.26), ("E4", 0.256), ("E4", 0.253),
  ("E4", 0.25), ("E4", 0.246), ("F4", 0.243), ("E4", 0.24), ("D#4", 0.237), ("E4", 0.466),
  ("A4", 0.455), ("C5", 0.879), ("D5", 0.214), ("D5", 0.211), ("D5", 0.209), ("D5", 0.207),
  ("D5", 0.204), ("C5", 0.202), ("B4", 0.2), ("D5", 0.198), ("C5", 0.196), ("C5", 0.194),
  ("C5", 0.192), ("C5", 0.19), ("C5", 0.188), ("B4", 0.187), ("A4", 0.185), ("C5", 0.183),
  ("B4", 0.181), ("B4", 0.18), ("B4", 0.178), ("B4", 0.176), ("F#4", 0.348), ("B4", 0.342),
  ("G#4", 1.309), ("E4", 0.162), ("E4", 0.164), ("E4", 0.161), ("E4", 0.159), ("E4", 0.157),
  ("E4", 0.155), ("E4", 0.153), ("E4", 0.151), ("E4", 0.149), ("E4", 0.148), ("E4", 0.146),
  ("E4", 0.144), ("E4", 0.142), ("E4", 0.141), ("E4", 0.139), ("E4", 0.137), ("E4", 0.136),
  ("F4", 0.134), ("E4", 0.133), ("D#4", 0.131), ("E4", 0.259), ("A4", 0.127), ("C5", 0.126),
  ("E5", 0.248), ("A5", 0.243), ("C6", 0.472), ("D6", 0.115), ("C6", 0.115), ("B5", 0.114),
  ("D6", 0.114), ("C6", 0.113), ("B5", 0.113), ("A5", 0.112), ("C6", 0.111), ("B5", 0.111),
  ("A5", 0.11), ("G#5", 0.11), ("B5", 0.109), ("A5", 0.109), ("E5", 0.108), ("C5", 0.108),
  ("A4", 0.107), ("F5", 0.107), ("E5", 0.107), ("D5", 0.106), ("C5", 0.106), ("B4", 0.105),
  ("A4", 0.105), ("G#4", 0.104), ("B4", 0.104), ("A4", 0.496),
]

In [ ]:
# Precompute when each note starts, so on_frame knows when to drop each sphere.
cues = []
t = 0.0
for note, duration in sequence:
    cues.append((t, note))
    t += duration

# Time for a sphere to fall from the spawn height down to the pad. Each sphere
# is released exactly this early, so it reaches the pad on its note.
DROP_HEIGHT = 6.0
fall_time = math.sqrt(2 * DROP_HEIGHT / GRAVITY)

BALL_DIAMETER = 0.7

elapsed = 0.0
spawn_index = 0

def drop_ball(cue_time, note):
    # How far above the pad to start so the fall lands on the note. The opening
    # notes have less than a full fall_time of runway, so they start lower.
    remaining = max(0.0, cue_time - elapsed)
    height = min(DROP_HEIGHT, 0.5 * GRAVITY * remaining * remaining)
    ball = scene3d.Shapes.Sphere(diameter=BALL_DIAMETER, segments=12)
    ball.set_position(pitch_x(note), PAD_Y + height, 0)
    ball.set_color(pitch_color(note))
    scene.add(ball)          # physics is on -> it starts falling immediately

@scene.on_frame
def animate(dt):
    global elapsed, spawn_index
    elapsed += dt
    while spawn_index < len(cues) and elapsed >= cues[spawn_index][0] - fall_time:
        cue_time, note = cues[spawn_index]
        drop_ball(cue_time, note)
        spawn_index += 1

# The music is scheduled up front so its timing stays exact; the spheres are
# released to line up with it visually.
await audio.play_notes_async(sequence)
scene.run()